# 05. 음성 → 텍스트: Whisper STT

**실습 목표**  
음성 파일을 자동으로 받아 적고, 다국어 음성 인식 파이프라인을 체험한다.

**주요 Hugging Face 모델**  
`openai/whisper-small`

> 이 노트북은 **파인튜닝 없이 사전학습 모델을 추론에 활용**하는 실습이다.  
> RTX 4060 8GB 환경을 고려했으며, CUDA가 없으면 CPU로 자동 전환하도록 구성하였다.

In [2]:
# uv add transformers accelerate torch torchaudio librosa soundfile

In [1]:
import torch
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
DEVICE = 0 if torch.cuda.is_available() else -1
TORCH_DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", TORCH_DEVICE)

PyTorch: 2.14.0+cu126
CUDA available: True
device: cuda


In [2]:
from transformers import pipeline
import torch

dtype = torch.float16 if torch.cuda.is_available() else torch.float32
asr = pipeline(
    "automatic-speech-recognition",
    model="openai/whisper-small",
    device=DEVICE,
    dtype=dtype
)

config.json:   0%|          | 0.00/1.97k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  967MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/3.87k [00:00<?, ?B/s]

...

merges.txt:   0%|          | 0.00/494k [00:00<?, ?B/s]

normalizer.json:   0%|          | 0.00/52.7k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/34.6k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.19k [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/185k [00:00<?, ?B/s]

In [ ]:
# sudo apt update
# sudo apt install -y ffmpeg

In [ ]:
# 영어
# AUDIO_URL = "https://huggingface.co/datasets/Narsil/asr_dummy/resolve/main/mlk.flac"

# 한글  Bingsu/KSS_Dataset
AUDIO_URL = "https://datasets-server.huggingface.co/assets/Bingsu/KSS_Dataset/--/default/train/0/audio/audio.wav?Expires=1788812452&Signature=tgokuJQuLK~6aGF4z6~RTa9~p-s9Kj19~Sjn5ZfGCQdS4vFmtcIsuXHVIyHOh2SKJKjguKCKe9mkHKkKxMlKmV0z-rRrtQWHLp23fyq0xVf0QwxUUHE0M90K~sUaLs-HciW4mW0MuIjLv1o3kE34sKYOXdVQd9-ZZUubhVKpCwazlXTDVPxV3jHLBBWHOByC6eBYUJxvLbVR4ZI5HtjOcRALoJBBYRjpnoRr~IjVfDFnTjBNywDiC2CcI1D1ILFODtT-t2CULmlxY8oT~G~RQpcLeA1UceVw-U72io3jWReq4UWmcgTCWVJBj~fEgBnyIxRi1JkyQJP4XIZ8IjBDiA__&Key-Pair-Id=K3C0L9WB6U5DUC"
result = asr(AUDIO_URL)
print(result["text"])

 그는 괜찮은 척 하려고 애쓰는 것 같았다.


In [3]:
# 본인 음성 파일 사용 예
result = asr("audio/audio_ko.wav")
# result = asr("audio/audio_ko.wav", return_timestamps=True)
result
# print(result["text"])

[transformers] Using custom `forced_decoder_ids` from the (generation) config. This is deprecated in favor of the `task` and `language` flags/config options.
[transformers] Transcription using a multilingual Whisper will default to language detection followed by transcription instead of translation to English. This might be a breaking change for your use case. If you want to instead always translate your audio to English, make sure to pass `language='en'`. See https://github.com/huggingface/transformers/pull/28687 for more details.
[transformers] Passing `generation_config` together with generation-related arguments=({'begin_suppress_tokens', 'suppress_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer WhisperTokenizer. The clean_up_tokenization post-processing step is designed for WordPiece token

{'text': ' 그는 괜찮은 척 하려고 애쓰는 것 같았다.'}

## 확장 과제
- 한국어 음성 파일을 녹음해 인식한다.
- `return_timestamps=True`를 적용해 자막용 타임스탬프를 생성해본다.